In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import tensorflow as tf
from tensorflow.io.gfile import glob
import os
import matplotlib.pyplot as plt
import tensorflow.keras as keras
import tensorflow.keras.layers as layers
from sklearn.decomposition import PCA
from tensorflow.keras.preprocessing.image import load_img
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import shutil
import PIL
import tensorflow_probability as tfp
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
import os
### Setting my path to the input file
#os.chdir("/kaggle/input/gan-getting-started")
for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname, len(os.listdir(dirname)))
AUTOTUNE = tf.data.experimental.AUTOTUNE

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Training strategy (TPU or GPU)

Later I create the one for TPUs

In [ ]:
if len(tf.config.list_physical_devices()) >= 3:
    strategy = tf.distribute.MirroredStrategy()
    print("Number of devices", strategy.num_replicas_in_sync)    
else: 
    strategy = tf.distribute.get_strategy()
strategy

# Loading data
Using the TFRecord files

### 

In [ ]:
Monet_files = glob(str('/kaggle/input/gan-getting-started/monet_tfrec/*.tfrec'))
print('Monet TFRecord Files:', len(Monet_files))
Photo_files = glob(str('/kaggle/input/dataset/data/photo_tfrec/*.tfrec'))
print('Photo TFRecord Files:', len(Photo_files))

### TFRecord reading to get feature names
To decode the TFRecords you need to specify the feature name and type in form of a dictionary

In [ ]:
raw_dataset = tf.data.TFRecordDataset(Monet_files[0])
for raw_record in raw_dataset.take(1):
    example = tf.train.Example()
    example.ParseFromString(raw_record.numpy())
    Monet_features = [i for i in example.features.feature]

raw_dataset = tf.data.TFRecordDataset(Photo_files[0])
for raw_record in raw_dataset.take(1):
    example = tf.train.Example()
    example.ParseFromString(raw_record.numpy())
    Photo_features = [i for i in example.features.feature]

print('Monet tfrecord Features:', Monet_features)
print('Photo tfrecord Features:', Photo_features)

In [ ]:
IMAGE_SIZE = [256, 256]

def decode_image(image):
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.convert_image_dtype(image, tf.float32)
    image = tf.reshape(image, [*IMAGE_SIZE, 3])
    return image

def read_tfrecord(example):
    tfrecord_format = {
        "image_name": tf.io.FixedLenFeature([], tf.string),
        "image": tf.io.FixedLenFeature([], tf.string),
        "target": tf.io.FixedLenFeature([], tf.string)
    }
    example = tf.io.parse_single_example(example, tfrecord_format)
    image = decode_image(example['image'])
    return image

def load_dataset(filenames, labeled=True, ordered=False, repeat = False):
    dataset = tf.data.TFRecordDataset(filenames)
    dataset = dataset.map(read_tfrecord, num_parallel_calls=AUTOTUNE)
    if repeat:
        dataset = dataset.repeat(count = 20)
    dataset = dataset.shuffle(1000)
    dataset = dataset.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)
    return dataset

## Image generators

In [ ]:
batch_size = 32
photo_ds = load_dataset(Photo_files, labeled = True).batch(batch_size)
monet_ds = load_dataset(Monet_files, labeled = True, repeat = True).batch(batch_size)
example_monet = next(iter(monet_ds))
example_photo = next(iter(photo_ds))

# Visualize categories

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(15, 6))
fig.suptitle('Image Categories', fontsize=16)

for i in range(0,5):
    ax = axes[0, i]
    ax.imshow(example_monet[i])
    ax.axis('off')

for i in range(0,5):
    ax = axes[1, i]
    ax.imshow(example_photo[i])
    ax.axis('off')

axes[0, 2].set_title('Monet', size='large', loc='center')
axes[1, 2].set_title('Photo', size='large', loc='center')
plt.tight_layout()  # Adjust layout to make room for the main title
plt.show()

# Basic blocks

## Downsample block

In [ ]:
class Downsample(keras.layers.Layer):
    def __init__(self,filters, kernel_size = (2,2), strides=(2, 2), padding='same', activation = 'leaky_relu', **kwargs):
        
        super(Downsample, self).__init__(**kwargs)
        self.initializer = tf.random_normal_initializer(0., 0.02)
        self.gamma_init = keras.initializers.RandomNormal(mean=0.0, stddev=0.02)
        self.conv = layers.Conv2D(filters, kernel_size = kernel_size, 
                                  strides = strides, padding = padding, kernel_initializer = self.initializer)
        self.bn = layers.BatchNormalization()
        self.activation = layers.Activation(activation = activation)
        
    def call(self, inputs,training = False):
        x = self.conv(inputs)
        x = self.bn(x, training = training)
        x = self.activation(x)
        
        return x
    
    def get_config(self):
        config = super(Downsample, self).get_config()
        config.update({
            'filters': self.conv.filters,
            'kernel_size': self.conv.kernel_size,
            'strides': self.conv.strides,
            'padding': self.conv.padding,
            'activation': self.activation.activation,
        })
        return config

In [ ]:
def apply_pca_and_visualize(convolved_images):
    # Flatten the convolved images
    shape = convolved_images.shape
    normalized_image = convolved_images / 255.0
    reshaped_image = tf.reshape(normalized_image, (-1, shape[-1]))
    reshaped_array = reshaped_image.numpy()
    # Apply PCA
    pca = PCA(n_components=1)
    pca_result = pca.fit_transform(reshaped_array)
    # Reshape back
    pca_image_reshaped = pca_result.reshape(shape[0], shape[1], 1)
    return pca_image_reshaped

In [ ]:
filters = [32,64,128,256]
convolutions = []
convolutions.append(example_monet)
for count,element in enumerate(filters):
    Block = Downsample(element)
    convolutions.append(Block(convolutions[count]))

fig, axes = plt.subplots(nrows=1, ncols=6, figsize=(15, 3))
fig.suptitle('Downsampling', fontsize=16)

batch_number = 3
## Original visualization
ax = axes[0]
ax.imshow(convolutions[0][batch_number])
axes[0].set_title('Original Monet', size='large', loc='center')

# PCA of original
ax = axes[1]
ax.imshow(apply_pca_and_visualize(convolutions[0][batch_number])*255.0, cmap = 'winter')
axes[1].set_title('PCA', size='large', loc='center')

# 
for i in range(1, len(convolutions)):
    ax = axes[i+1]
    ax.imshow(apply_pca_and_visualize(convolutions[i][batch_number])*255.0, cmap = 'winter')
    axes[i+1].set_title(f'Downsample {convolutions[i].shape[-1]} filters', size='medium', loc='center')
    
plt.show()

## Residual Block

In [ ]:
class ResidualBlock(keras.layers.Layer):
    def __init__(self, filters = 512, kernel_size=(3, 3), strides=(1, 1), padding='same', activation = 'leaky_relu', **kwargs):
        super(ResidualBlock, self).__init__(**kwargs)
        self.conv1 = layers.Conv2D(filters, kernel_size, strides, padding)
        self.bn1 = layers.BatchNormalization()
        self.activation = layers.ReLU()
        self.conv2 = layers.Conv2D(filters, kernel_size, strides, padding)
        self.bn2 = layers.BatchNormalization()

    def call(self, inputs, training=False):
        x = self.conv1(inputs)
        x = self.bn1(x, training=training)
        x = self.activation(x)
        x = self.conv2(x)
        x = self.bn2(x)
        return layers.Add()([inputs, x])

## Upsample block

In [ ]:
class Upsample(keras.layers.Layer):
    def __init__(self,filters, kernel_size = (2,2), strides=(2, 2), padding='same', activation = 'leaky_relu', dropout = 0.2,**kwargs):
        
        super(Upsample, self).__init__(**kwargs)
        self.initializer = tf.random_normal_initializer(0., 0.02)
        self.conv = layers.Conv2DTranspose(filters, kernel_size = kernel_size, 
                                           strides = strides, padding = padding, kernel_initializer = self.initializer)
        self.bn = layers.BatchNormalization()
        self.dropout = layers.Dropout(dropout)
        self.activation = layers.Activation(activation = activation)
    
    def call(self, inputs,training = False, dropout = False):
        x = self.conv(inputs)
        x = self.bn(x, training = training)
        if dropout:
            x = self.dropout(x)
        x = self.activation(x)
        
        return x
    
    def get_config(self):
        config = super(Upsample, self).get_config()
        config.update({
            'filters': self.conv.filters,
            'kernel_size': self.conv.kernel_size,
            'strides': self.conv.strides,
            'padding': self.conv.padding,
            'activation': self.activation.activation,
        })
        return config

In [ ]:
upsample_filters = [512,256,128,3]
upsample_test = []
upsample_test.append(convolutions[-1])
for count,element in enumerate(upsample_filters):
    upsample_block = Upsample(element)
    upsample_test.append(upsample_block(upsample_test[count], dropout = True, training = False))
    
fig, axes = plt.subplots(nrows=1, ncols=6, figsize=(15, 3))
fig.suptitle('Upsampling', fontsize=16)

batch_number = 3
 
for i in range(0, len(upsample_test)-1):
    ax = axes[i]
    ax.imshow(apply_pca_and_visualize(upsample_test[i][batch_number])*255.0, cmap = 'winter')
    axes[i].set_title(f'Upsample {upsample_test[i].shape[-1]} filters', size='medium', loc='center')

# PCA of the predicted original
ax = axes[4]
ax.imshow(apply_pca_and_visualize(upsample_test[-1][batch_number])*255.0, cmap = 'winter')
axes[4].set_title('PCA', size='large', loc='center')

# PCA of the predicted original
ax = axes[5]
ax.imshow(upsample_test[-1][batch_number]*255.0)
axes[5].set_title('Original', size='large', loc='center')

plt.show()

# Defining generator

In [ ]:
class Generator(keras.Model):
    def __init__(self, name = 'Generator',**kwargs):
        super(Generator, self).__init__(name = name, **kwargs)
        
        ### Downsampling layers
        self.downsample1 = Downsample(64, name = 'Donwsample_same', strides = (1,1), kernel_size = (5,5))   ## (bs,256,256,64)
        self.downsample2 = Downsample(128, name = 'Donwsample_128')   ## (bs,128,128,128)
        self.downsample3 = Downsample(128, name = 'Donwsample_64')  ## (bs,64,64,128)
        self.downsample4 = Downsample(256, name = 'Donwsample_32')  ## (bs,32,32,256)
        self.downsample5 = Downsample(512, name = 'Donwsample_16')  ## (bs,16,16,512)
        self.downsample6 = Downsample(512, name = 'Donwsample_8')  ## (bs,8,8,512)
        self.downsample7 = Downsample(512, name = 'Donwsample_4')  ## (bs,4,4,512)
        
        self.residual1 = ResidualBlock()
        self.residual2 = ResidualBlock()
        self.residual3 = ResidualBlock()
        
        ## Upsampling layers
        self.upsample1 = Upsample(512, name = 'Upsample_8') ## (bs,8,8,512)
        self.upsample2 = Upsample(512, name = 'Upsample_16') ## (bs,16,16,512)
        self.upsample3 = Upsample(256,name = 'Upsample_32' ) ## (bs,32,32,512)
        self.upsample4 = Upsample(128, name = 'Upsample_64') ## (bs,64,64,256)
        self.upsample5 = Upsample(128, name = 'Upsample_128') ## (bs,128,128,128)
        self.upsample6 = Upsample(64, name = 'Upsample_256') ## (bs,256,256,64)
        self.final_upsample = Upsample(3, activation = 'tanh', name = 'Output_layer', strides = (1,1)) # (bs,256,256,3)
        
        
    def call(self, inputs, training = False):
        ### Downsampling
        d1 = self.downsample1(inputs, training = training)
        d2 = self.downsample2(d1, training = training)
        d3 = self.downsample3(d2, training = training)
        d4 = self.downsample4(d3, training = training)
        d5 = self.downsample5(d4, training = training)
        d6 = self.downsample6(d5, training = training)
        d7 = self.downsample7(d6, training = training)
        
        r1 = self.residual1(d7, training = training)
        r2 = self.residual2(r1, training = training)
        r3 = self.residual3(r2, training = training)

        ## Upsampling
        u1 = self.upsample1(r3, training = training)
        u1_concat = tf.concat([u1, d6], axis=-1)
        u2 = self.upsample2(u1_concat, training = training)
        u2_concat = tf.concat([u2, d5], axis=-1)
        u3 = self.upsample3(u2_concat, training = training)
        u3_concat = tf.concat([u3, d4], axis=-1)
        u4 = self.upsample4(u3_concat, training = training)
        u4_concat = tf.concat([u4, d3], axis=-1)
        u5 = self.upsample5(u4_concat, training = training)
        u5_concat = tf.concat([u5, d2], axis=-1)
        u6 = self.upsample6(u5_concat, training = training)
        u6_concat = tf.concat([u6, d1], axis=-1)
        x = self.final_upsample(u6_concat, training = training)
        return x

## Compiling the Generator
Just to make sure everything is working properly

In [ ]:
g = Generator()
g.compile(optimizer = 'adam', loss = 'mse')
x = g(example_photo, training = True)  ## to build the generator you need to pass an input trough the network
g.summary()

## Visualize the generator
Since its not trained it will only show blank images. Just to make sure the shape of the generated images are correct

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(15, 6))
fig.suptitle('Monet Generator', fontsize=16)
ax = axes[0]
ax.imshow(example_photo[2])
axes[0].set_title('Photo', size='large', loc='center')
ax = axes[1]
ax.imshow(x[2])
axes[1].set_title('Photo as a Monet', size='large', loc='center')
plt.show()

# Defining the discriminator

In [ ]:
class Discriminator(keras.Model):
    def __init__(self, name = 'Discriminator', dense = 64, activation = 'leaky_relu', **kwargs):
        super(Discriminator, self).__init__(name = name, **kwargs)
        self.initializer = tf.random_normal_initializer(0., 0.02)
        self.downsample1 = Downsample(64, name = 'Downsample_128')   ## (bs,128,128,64)
        self.downsample2 = Downsample(128, name = 'Downsample_64')  ## (bs, 64,64,128)
        self.downsample3 = Downsample(256, name = 'Downsample_32')  ## (bs, 32,32,256)
        self.downsample4 = Downsample(256, name = 'Downsample_16')  ## (bs, 32,32,256)
        self.zeropad = layers.ZeroPadding2D(name = 'Zero_Padding')
        self.downsample5 = Downsample(512, strides = (1,1), kernel_size = (2,2), name = 'Downsample_stride_1')  ## (bs,32,32,512)
        self.flatten = layers.Flatten(name = 'Flatten_layer')
        self.dense = layers.Dense(dense, name = f'Hiden_layer_{dense}')
        self.dropout = layers.Dropout(0.2, name = 'Dropout')
        self.activation = layers.Activation(activation,name =  f'{activation}')
        self.last = layers.Dense(1, activation = 'sigmoid')
        
    def call(self,inputs, training = False):
        x = self.downsample1(inputs, training = training)
        x = self.downsample2(x, training = training)
        x = self.downsample3(x, training = training)
        x = self.downsample4(x, training = training)
        x = self.zeropad(x)
        x = self.downsample5(x, training =training)
        x = self.flatten(x)
        x = self.dense(x)
        x = self.dropout(x)
        x = self.activation(x)
        x = self.last(x)
        return x

## Compiling the discriminator

In [ ]:
d = Discriminator()
d.compile(optimizer = 'adam', loss = 'mse')
y = d(example_monet, training = False)
d.summary()

## Explaining each loss function
Well go explaining each function and testing it to give an example of how they work. To better understand them. There's $x$ and $y$; the output of and untrained generator and discriminator respectivelly
### Generator Loss
How good the Generator is able to fool the discriminator  
Mathematically : $L_G = -log(D(G((z))$ where $G(z)$ is the generated image and 
$(D(G(z))$ is the probability that the generated image is real  
In the generator loss, we'll compare the output of the discriminator $D(G(z))$ with a tensor with equal shape filled with ones. This tensor represents the goal of the Generator, create 'real' examples that in our case are labeled with 1

In [ ]:
def generator_loss(disc_output):
    ### BinaryCrossentropy cause were using only two categories (real or fake)
    ### from_logits to avoid passing the discriminator output trough a sigmoid activation
    return keras.losses.BinaryCrossentropy(from_logits=False)(tf.ones_like(disc_output), disc_output)

print('Generator loss: ', generator_loss(y))

### Discriminator Loss
How well the discriminator is able to distinguish between real and fake images.  
Mathematically $L_D = -log(D(x)) + -log((D(G(z))$ where we're going to compare the discriminator output when trying to classify real images with a tensor filled with ones, and the oposite, compare the discriminator output when classifying generated images with a tensor filled with zeros.

In [ ]:
def discriminator_loss(real_output, fake_output):
    ### D(x) when classifying real examples
    real_loss = tf.keras.losses.BinaryCrossentropy(from_logits=False)(tf.ones_like(real_output), real_output)
    ### D(y) when classiying fake examples
    fake_loss = tf.keras.losses.BinaryCrossentropy(from_logits=False)(tf.zeros_like(fake_output), fake_output)
    return (real_loss + fake_loss)*0.5

print('Discriminator loss: ', discriminator_loss(y,y))

### Cycle Consistency Loss
Ensures that the process of translating and image from one domain and back, should result in the original image. $F(G(x)) ≈ x$ and $G(F(y)) ≈ y$  
Mathematically: $L_{cycle} = ||F(G(x)) - x||_1 + ||G(F(y)) - y||_1$
In simpler words, this compares original image against the full cycled image and computes the average absolute difference between them, this for both categories.  
In the paper they add $\lambda$, this is a wight factor that controls the importance of the objective. In this case ${\lambda}_{cycle}$ multiplies $L_{cycle}$ to denote how important for our purpose is the cycle consistency

In [ ]:
### Compares the original image against the cycled one
def cycle_loss(real_image, cycled_image, lambda_cycle=10):
    return lambda_cycle * tf.reduce_mean(tf.abs(real_image - cycled_image))
### When comparing the same images, the cycle loss should be 0
print('Cycle loss: ', cycle_loss(x[0],x[0]))

### Identity loss
Helps preserve the color composition between input and output $G(x) ≈ x$, this doesn't compared the full cycled image, this compares input and output from a generator that translates to the same domain.  
Mathematically: $L_{Identity} = ||G(x) - x||_1 + ||F(y) - y||_1$
Also we need to multiply by $\lambda$, can be different than ${\lambda}_{cycle}$ . In this case ${\lambda}_{Identity}$ multiplies $L_{Identity}$ to denote how important for our purpose is the color composition

In [ ]:
def identity_loss(real_image, same_image, lambda_identity=5):
    ## Real image is the original image from the source domain
    ## Same image: image processed by the generator but in the same domain
    return lambda_identity * tf.reduce_mean(tf.abs(real_image - same_image))

### When comparing the same images, the cycle loss should be 0
print('Identity Loss: ', identity_loss(x[0],x[0]))

## Building the CycleGAN

In [ ]:
import tensorflow as tf
import tensorflow.keras as keras
import numpy as np
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from tensorflow.keras.preprocessing import image
from scipy.linalg import sqrtm
# from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2
# 创建CycleGAN网络模型
class CycleGAN(keras.Model):
    def __init__(self, name='CycleGAN', lambda_cycle=10, lambda_identity=5, trainable=False, dtype='float32'):
        super(CycleGAN, self).__init__(name=name, trainable=trainable, dtype=dtype)

        self.monet_generator_ = Generator(name='Monet_Generator')
        self.monet_discriminator_ = Discriminator(name='Monet_Discriminator')
        self.photo_generator_ = Generator(name='Photo_Generator')
        self.photo_discriminator_ = Discriminator(name='Photo_Discriminator')
        self.lambda_cycle = lambda_cycle
        self.lambda_identity = lambda_identity
        # 加载预训练的 VGG16 模型
        # self.vgg = VGG16(weights='imagenet', include_top=False, input_shape=(256, 256, 3))
        # self.vgg.trainable = False
        self.mobilenet = MobileNetV2(weights='imagenet', include_top=False, input_shape=(256, 256, 3))
        self.mobilenet.trainable = False

    def compile(self, m_gen_optimizer, p_gen_optimizer, m_disc_optimizer, p_disc_optimizer, **kwargs):
        super(CycleGAN, self).compile(**kwargs)
        # 初始化生成器、判别器m的优化器
        self.m_gen_optimizer = m_gen_optimizer
        self.m_disc_optimizer = m_disc_optimizer
        # 初始化生成器、判别器p的优化器
        self.p_gen_optimizer = p_gen_optimizer
        self.p_disc_optimizer = p_disc_optimizer
        # 二分类交叉熵
        self.bce_ = tf.keras.losses.BinaryCrossentropy(from_logits=False)


    # 生成器损失
    def generator_loss_(self, disc_output):
        return self.bce_(tf.ones_like(disc_output), disc_output)

    # 判别器损失
    def discriminator_loss_(self, real_output, fake_output):
        real_loss = self.bce_(tf.ones_like(real_output), real_output)
        fake_loss = self.bce_(tf.zeros_like(fake_output), fake_output)
        return (real_loss + fake_loss)*0.5

    # 循环网络损失
    def cycle_loss_(self, real_image, cycled_image):
        return self.lambda_cycle * tf.reduce_mean(tf.abs(real_image - cycled_image))

    # 身份损失
    def identity_loss_(self, real_image, same_image):
        return self.lambda_identity * tf.reduce_mean(tf.abs(real_image - same_image))

    # 计算感知损失
    def perceptual_loss_(self, real_image, generated_image):
        # 通过vgg预训练网络计算L1 loss
        real_features = self.mobilenet(real_image)
        generated_features = self.mobilenet(generated_image)
        loss = tf.reduce_mean(tf.abs(real_features - generated_features))
        return loss

    # 训练过程
    def train_step(self, batch_data):
        real_monet, real_photo = batch_data
        ### Persistent True to compute multiple gradients for the same tape
        with tf.GradientTape(persistent=True) as tape:
            # 前向传播
            fake_monet = self.monet_generator_(real_photo, training=True)  ## Photo to fake Monet
            cycled_photo = self.photo_generator_(fake_monet, training=True)  ## fake Monet to photo
            fake_photo = self.photo_generator_(real_monet, training=True)  ## Monet to fake photo
            cycled_monet = self.monet_generator_(fake_photo, training=True)  ## fake Photo to Monet

            ### Identity mapping (G(x) to x, F(y) to y)
            same_monet = self.monet_generator_(real_monet, training=True)  ## Monet to monet
            same_photo = self.photo_generator_(real_photo, training=True)  ## Photo to photo

            # 判别器预测，4个判别器
            disc_real_monet = self.monet_discriminator_(real_monet, training=True)  ## Monet is real Monet
            disc_fake_monet = self.monet_discriminator_(fake_monet, training=True)  ## Fake Monet is real Monet
            disc_real_photo = self.photo_discriminator_(real_photo, training=True)  ## Photo is a real photo
            disc_fake_photo = self.photo_discriminator_(fake_photo, training=True)  ## Fake Photo is a real photo

            # 计算损失
            ## Generators
            gen_monet_loss = self.generator_loss_(disc_fake_monet)
            gen_photo_loss = self.generator_loss_(disc_fake_photo)
            ## Discriminators
            disc_monet_loss = self.discriminator_loss_(disc_real_monet, disc_fake_monet)
            disc_photo_loss = self.discriminator_loss_(disc_real_photo, disc_fake_photo)
            
            ### Cycle
            cycle_loss = self.cycle_loss_(real_monet, cycled_monet) + self.cycle_loss_(real_photo, cycled_photo)
            ### Identity
            identity_loss = self.identity_loss_(real_monet, same_monet) + self.identity_loss_(real_photo, same_photo)

            ### Perceptual
            perceptual_loss_monet = self.perceptual_loss_(real_monet, fake_monet)
            perceptual_loss_photo = self.perceptual_loss_(real_photo, fake_photo)

#             ### Generators total loss
#             total_gen_monet_loss = gen_monet_loss + cycle_loss + identity_loss
#             total_gen_photo_loss = gen_photo_loss + cycle_loss + identity_loss

            # perceptual_loss
            total_gen_monet_loss = gen_monet_loss + cycle_loss + identity_loss + perceptual_loss_monet
            total_gen_photo_loss = gen_photo_loss + cycle_loss + identity_loss + perceptual_loss_photo

        # Gradients梯度
        # 计算对应损失函数以及模型可训练参数（卷积层、批归一化层、激活函数等网络层的权重和偏置）
        monet_generator_gradients = tape.gradient(total_gen_monet_loss, self.monet_generator_.trainable_variables)
        photo_generator_gradients = tape.gradient(total_gen_photo_loss, self.photo_generator_.trainable_variables)
        monet_discriminator_gradients = tape.gradient(disc_monet_loss, self.monet_discriminator_.trainable_variables)
        photo_discriminator_gradients = tape.gradient(disc_photo_loss, self.photo_discriminator_.trainable_variables)

        # gradients梯度的反向传播
        # 使用zip将梯度和训练参数打包，并对训练参数进行梯度下降操作
        self.m_gen_optimizer.apply_gradients(zip(monet_generator_gradients, self.monet_generator_.trainable_variables))
        self.p_gen_optimizer.apply_gradients(zip(photo_generator_gradients, self.photo_generator_.trainable_variables))
        self.m_disc_optimizer.apply_gradients(
            zip(monet_discriminator_gradients, self.monet_discriminator_.trainable_variables))
        self.p_disc_optimizer.apply_gradients(
            zip(photo_discriminator_gradients, self.photo_discriminator_.trainable_variables))

        # #
        # # 计算FID
        # fid_score = self.calculate_fid(real_monet, fake_monet) + self.calculate_fid(real_photo, fake_photo)
        # # 计算IS
        # is_score = self.calculate_is(fake_monet) + self.calculate_is(fake_photo)

        return {
            "monet_gen_loss": total_gen_monet_loss,
            "photo_gen_loss": total_gen_photo_loss,
            "monet_disc_loss": disc_monet_loss,
            "photo_disc_loss": disc_photo_loss
            }

# Training
## Callbacks
Setting up callbacks to visualize the training process. Could use wandb but it's better to have everything on the notebook

In [ ]:
class callbacks(keras.callbacks.Callback):
    def __init__(self, fid_interval, generator, example_photo, example_monet):
        self.fid_interval = fid_interval
        self.generator = generator
        self.example_photo = example_photo
        self.example_monet = example_monet
        self.fid_values_monet = []
        self.fid_values_photo = []
        self.losses = []


    def on_epoch_end(self, epoch, logs={}):
        ### Every 5 epochs plot the generator
        randomint = np.random.randint(0, 15, 1)[0]

        # 加入损失
        self.losses.append(logs)
        # print(epoch, logs)
        
        if epoch % self.fid_interval == 0:
            fake_monet = self.generator.monet_generator_(self.example_photo, training=True)
            fake_photo = self.generator.photo_generator_(self.example_monet, training=True)
            fid_monet_value = self.calculate_fid(fake_monet, self.example_monet)
            fid_photo_value = self.calculate_fid(fake_photo, self.example_photo)
            print(f"fid_monet:{fid_monet_value},fid_photo:{fid_photo_value}")
            self.fid_values_monet.append(fid_monet_value)
            self.fid_values_photo.append(fid_photo_value)
            # save model
            tf.saved_model.save(cycle_gan, f'mymodel/{epoch}/')


        if epoch != 0:
            fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(15, 4))
            fig.suptitle(f'Generator evolution epoch #{epoch}', fontsize=16)
            ax = axes[0]
            ax.imshow(example_photo[randomint])
            axes[0].set_title('Original Photo', size='large', loc='center')
            ax.axis('off')
            ax = axes[1]
            ax.imshow(cycle_gan.monet_generator_(example_photo)[randomint])
            axes[1].set_title('Photo to Monet', size='large', loc='center')
            ax.axis('off')
            ax = axes[2]
            ax.imshow(example_monet[randomint])
            axes[2].set_title('Original Monet', size='large', loc='center')
            ax.axis('off')
            ax = axes[3]
            ax.imshow(cycle_gan.photo_generator_(example_monet)[randomint])
            axes[3].set_title('Monet to photo', size='large', loc='center')
            ax.axis('off')
            plt.savefig(f'{epoch}.png')
            plt.show()
            # save model
            # 保存模型
            tf.saved_model.save(cycle_gan, f'mymodel/{epoch}/')
            
        if epoch == 0:
            fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(15, 4))
            fig.suptitle(f'Starting point', fontsize=16)
            ax = axes[0]
            ax.imshow(example_photo[randomint])
            axes[0].set_title('Original Photo', size='large', loc='center')
            ax.axis('off')
            ax = axes[1]
            ax.imshow(cycle_gan.monet_generator_(example_photo)[randomint])
            axes[1].set_title('Photo to Monet', size='large', loc='center')
            ax.axis('off')
            ax = axes[2]
            ax.imshow(example_monet[randomint])
            axes[2].set_title('Original Monet', size='large', loc='center')
            ax.axis('off')
            ax = axes[3]
            ax.imshow(cycle_gan.photo_generator_(example_monet)[randomint])
            axes[3].set_title('Monet to photo', size='large', loc='center')
            ax.axis('off')
            plt.savefig(f'{epoch}.png')
            plt.show()

    def calculate_fid(self, real_images, fake_images):
        # 调整图像大小为299x299
        real_images = tf.image.resize(real_images, (299, 299))
        fake_images = tf.image.resize(fake_images, (299, 299))

        # 图像预处理
        real_images = tf.keras.applications.inception_v3.preprocess_input(real_images)
        fake_images = tf.keras.applications.inception_v3.preprocess_input(fake_images)

        # 加载 InceptionV3 模型
        model = tf.keras.applications.InceptionV3(include_top=False, pooling='avg', input_shape=(299, 299, 3))

        # 提取特征
        real_features = model(real_images)
        fake_features = model(fake_images)

        # 计算 FID
        mu_real = tf.reduce_mean(real_features, axis=0)
        mu_fake = tf.reduce_mean(fake_features, axis=0)

        # 计算样本协方差矩阵
        cov_real = tfp.stats.covariance(real_features, sample_axis=0)
        cov_fake = tfp.stats.covariance(fake_features, sample_axis=0)

        diff = mu_real - mu_fake
        # 使用数值稳定的方法计算FID
        cov_sqrtm = tf.linalg.sqrtm(tf.linalg.matmul(cov_real, cov_fake))
        trace = tf.linalg.trace(cov_real + cov_fake - 2 * cov_sqrtm)
        fid = tf.sqrt(tf.reduce_sum(diff ** 2) + tf.clip_by_value(trace, 0, 1e9))

        return fid

## Optimizers
Not sure what to use, using Adam

In [ ]:
with strategy.scope():
    m_gen_opt = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
    p_gen_opt = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
    m_disc_opt = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
    p_disc_opt = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

## Creating the model
Mirrored strategy for using multiple GPU's

In [ ]:
with strategy.scope():
    cycle_gan = CycleGAN()
    cycle_gan.compile(m_gen_opt, p_gen_opt, m_disc_opt, p_disc_opt)
cycle_gan.summary()
# 设置混合精度策略
tf.keras.mixed_precision.set_global_policy('mixed_float16')

开始训练Cycle GAN模型，并对训练结果进行可视化


In [ ]:
# 创建回调函数
fid_interval = 5  # 每隔5个epoch计算一次FID
callback = callbacks(fid_interval, cycle_gan, example_photo, example_monet)

# 训练模型
cycle_gan.fit(tf.data.Dataset.zip((monet_ds, photo_ds)), epochs=30, verbose=1, callbacks=callback)

# 对部分的图片进行莫奈风格转换后显示
fig, axes = plt.subplots(nrows=4, ncols=5, figsize=(15, 6))
fig.suptitle('Results', fontsize=16)

monet_results = cycle_gan.monet_generator_(example_photo)[0:10]
# 显示原始照片
for i in range(10):
    ax = axes[i // 5, i % 5]
    ax.imshow(example_photo[i])
    ax.axis('off')

# 显示莫奈风格的图像
for count, element in enumerate(monet_results):
    ax = axes[2 + count // 5, count % 5]
    ax.imshow(element)
    ax.axis('off')

axes[0, 2].set_title('Photos', size='large', loc='center')
axes[2, 2].set_title('Photos as Monet', size='large', loc='center')
plt.tight_layout()  # Adjust layout to make room for the main title
plt.savefig(f'final.png')
plt.show()

损失函数的可视化

In [ ]:
# 可视化损失函数
gen_monet_losses = [epoch['monet_gen_loss'] for epoch in callback.losses]
photo_gen_losses = [epoch['photo_gen_loss'] for epoch in callback.losses]
monet_disc_losses = [epoch['monet_disc_loss'] for epoch in callback.losses]
photo_disc_losses = [epoch['photo_disc_loss'] for epoch in callback.losses]

epochs = range(1, len(callback.losses) + 1)

plt.figure(figsize=(10, 5))
plt.plot(epochs, gen_monet_losses)
plt.plot(epochs, photo_gen_losses)
plt.plot(epochs, monet_disc_losses)
plt.plot(epochs, photo_disc_losses)
plt.scatter(epochs, gen_monet_losses, label='Monet Generator Loss')
plt.scatter(epochs, photo_gen_losses, label='Photo Generator Loss')
plt.scatter(epochs, monet_disc_losses, label='Monet Discriminator Loss')
plt.scatter(epochs, photo_disc_losses, label='Photo Discriminator Loss')
plt.title('CycleGAN Training Losses')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.savefig("loss.png")
plt.show()


可视化FID和IS指标

In [ ]:
# 创建一个新的图像
plt.figure(figsize=(10, 6))
epochs = range(1, len(callback.losses) + 1, 5)
# 绘制莫奈风格图像的FID值
plt.plot(epochs, callback.fid_values_monet, label='Monet Style')
plt.scatter(epochs, callback.fid_values_monet, label='Monet Style')
# 绘制照片的FID值
plt.plot(epochs, callback.fid_values_photo, label='Photo Style')
plt.scatter(epochs, callback.fid_values_photo, label='Photo Style')

plt.title('CycleGAN FID Score')
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.legend()
plt.savefig("FID_Score.png")
plt.show()

# 保存模型
tf.saved_model.save(cycle_gan, 'mymodel/')